# LlamaCloud Extract V2: Complete Walkthrough

This notebook demonstrates the full capabilities of LlamaCloud's Extract V2 API, from simple receipt extraction to advanced features like citations with bounding box visualization, confidence scores, and agentic extraction.

## Section 1: Setup

In [ ]:
%pip install -q "llama-cloud @ git+https://github.com/run-llama/llama-cloud-py.git@release-please--branches--main--changes--next"
%pip install -q pdf2image Pillow matplotlib python-dotenv

In [ ]:
import os
import json
import time
from dotenv import load_dotenv
from llama_cloud import LlamaCloud

load_dotenv()

client = LlamaCloud(
    api_key=os.environ["LLAMA_CLOUD_API_KEY"],
    # base_url="https://api.cloud.llamaindex.ai",  # default, can omit
)
print("Client initialized.")

### Upload test files

We upload three documents that represent common extraction scenarios: a receipt, a US patent (the Transformer paper, 16 pages), and a SaaS pitch deck slide.

In [ ]:
# Upload all three test files
test_files = {
    "receipt": "test-files/receipt/noisebridge_receipt.pdf",
    "patent": "test-files/patent/US10452978_transformer.pdf",
    "slide": "test-files/slide/saas_slide.pdf",
}

file_ids = {}
for name, path in test_files.items():
    with open(path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="extract")
    file_ids[name] = uploaded.id
    print(f"{name}: {uploaded.name} -> {uploaded.id}")

print("\nAll files uploaded.")

### Helper: wait for job completion

The SDK provides a built-in `wait_for_completion` method that polls until the job reaches a terminal status. We wrap it here for convenience.

In [ ]:
def wait_for_completion(client, job_id, verbose=True):
    """Poll an extract job until it completes. Returns the finished job."""
    completed = client.extract.wait_for_completion(job_id, verbose=verbose)
    print(f"Job {completed.id}: {completed.status}")
    return completed

---
## Section 2: Quick Start. Receipt Extraction

Extract structured data from a simple one-page receipt using a JSON Schema.

In [ ]:
from pdf2image import convert_from_path
import matplotlib.pyplot as plt

receipt_pages = convert_from_path("test-files/receipt/noisebridge_receipt.pdf", dpi=150)
fig, ax = plt.subplots(figsize=(8, 10))
ax.imshow(receipt_pages[0])
ax.set_title("Source Document: Noisebridge Receipt")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Load the receipt schema
with open("test-files/receipt/schema.json") as f:
    receipt_schema = json.load(f)

print(json.dumps(receipt_schema, indent=2))

In [ ]:
# Create an extract job
receipt_job = client.extract.create(
    document_input_value=file_ids["receipt"],
    configuration={
        "data_schema": receipt_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
    },
)
print(f"Created job: {receipt_job.id} (status: {receipt_job.status})")

In [ ]:
# Wait for completion and show results
receipt_job = wait_for_completion(client, receipt_job.id)

print("\n--- Extract Result ---")
print(json.dumps(receipt_job.extract_result, indent=2))

---
## Section 3: Complex Document. US Patent (16 pages)

Extract 15 fields from the Transformer patent (US10452978), including arrays of inventors, references, and classification codes.

In [ ]:
patent_pages = convert_from_path("test-files/patent/US10452978_transformer.pdf", dpi=150, first_page=1, last_page=1)
fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(patent_pages[0])
ax.set_title("Source Document: US10452978 (Transformer Patent) - Page 1 of 16")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Load the patent schema
with open("test-files/patent/schema.json") as f:
    patent_schema = json.load(f)

print(f"Schema has {len(patent_schema['properties'])} fields")
print(f"\nField names: {list(patent_schema['properties'].keys())}")

In [ ]:
# Extract the patent
patent_job = client.extract.create(
    document_input_value=file_ids["patent"],
    configuration={
        "data_schema": patent_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
    },
)

patent_job = wait_for_completion(client, patent_job.id)

print("\n--- Extract Result (all 15 fields) ---")
result = patent_job.extract_result
for field, value in result.items():
    display_val = str(value)[:80]
    if isinstance(value, list):
        display_val = f"[{len(value)} items] {display_val}"
    print(f"  {field}: {display_val}")

---
## Section 4: Citations & Bounding Boxes

Enable `cite_sources` to get per-field citations with page numbers, matching text, and bounding box coordinates. Then visualize them on the PDF page images.

In [ ]:
# Create an extract job with citations enabled (using the receipt)
citation_job = client.extract.create(
    document_input_value=file_ids["receipt"],
    configuration={
        "data_schema": receipt_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
        "cite_sources": True,
    },
)

citation_job = wait_for_completion(client, citation_job.id)
print(f"\nExtract result keys: {list(citation_job.extract_result.keys())}")

In [ ]:
# Fetch the job with expanded metadata to get per-field citations
citation_job_detail = client.extract.get(
    citation_job.id,
    expand=["extract_metadata"],
)

# Inspect the metadata structure
metadata = citation_job_detail.extract_metadata
print(f"extract_metadata type: {type(metadata)}")
print(f"field_metadata type: {type(metadata.field_metadata)}")

doc_metadata = metadata.field_metadata.document_metadata
print(f"\nFields with metadata: {len(doc_metadata)}")
print(f"Field names: {list(doc_metadata.keys())[:5]}...")

In [ ]:
# Print per-field citations
print("--- Per-Field Citations ---\n")
for field_name, field_meta in doc_metadata.items():
    if not isinstance(field_meta, dict):
        continue
    citations = field_meta.get("citation", [])
    if not citations:
        print(f"{field_name}: no citations")
        continue
    for cite in citations:
        page = cite.get("page")
        text = cite.get("matching_text", "")[:80]
        bboxes = cite.get("bounding_boxes", [])
        print(f"{field_name}:")
        print(f"  page: {page}")
        print(f"  matching_text: {text}")
        print(f"  bounding_boxes: {bboxes}")
        if cite.get("page_dimensions"):
            print(f"  page_dimensions: {cite['page_dimensions']}")
        print()

### Visualize bounding boxes on PDF pages

Draw red rectangles on the rendered PDF page images to show exactly where each extracted field was found.

In [ ]:
from pdf2image import convert_from_path
from PIL import ImageDraw
import matplotlib.pyplot as plt

# Convert PDF pages to images
pdf_pages = convert_from_path("test-files/receipt/noisebridge_receipt.pdf", dpi=150)
print(f"Rendered {len(pdf_pages)} pages")
print(f"Page 1 size: {pdf_pages[0].size} (width x height)")

In [ ]:
def draw_citation_boxes(page_img, field_name, citations):
    """
    Draw bounding boxes on a PDF page image for a given field's citations.
    
    Bounding box coordinates from the API are in PDF points (absolute),
    with page_dimensions providing the reference PDF page size.
    We scale from PDF coordinates to rendered image pixel coordinates.
    """
    img = page_img.copy()
    draw = ImageDraw.Draw(img)
    img_w, img_h = img.size

    for cite in citations:
        page_dims = cite.get("page_dimensions", {})
        pdf_w = page_dims.get("width", 612.0)
        pdf_h = page_dims.get("height", 792.0)

        scale_x = img_w / pdf_w
        scale_y = img_h / pdf_h

        for bbox in cite.get("bounding_boxes", []):
            x = bbox["x"] * scale_x
            y = bbox["y"] * scale_y
            w = bbox["w"] * scale_x
            h = bbox["h"] * scale_y

            draw.rectangle([x, y, x + w, y + h], outline="red", width=3)
            draw.text((x, max(0, y - 15)), field_name, fill="red")

    return img

In [ ]:
# Pick fields to visualize (choose ones with bounding boxes)
fields_to_show = []
for field_name, field_meta in doc_metadata.items():
    if not isinstance(field_meta, dict):
        continue
    citations = field_meta.get("citation", [])
    for cite in citations:
        if cite.get("bounding_boxes"):
            fields_to_show.append(field_name)
            break

# Show up to 4 fields
fields_to_show = fields_to_show[:4]
print(f"Visualizing {len(fields_to_show)} fields: {fields_to_show}")

In [ ]:
# Visualize each field's bounding boxes
fig, axes = plt.subplots(1, len(fields_to_show), figsize=(6 * len(fields_to_show), 10))
if len(fields_to_show) == 1:
    axes = [axes]

for ax, field_name in zip(axes, fields_to_show):
    field_meta = doc_metadata[field_name]
    citations = field_meta.get("citation", [])
    extracted_val = citation_job_detail.extract_result.get(field_name)

    # Determine which page to render (1-indexed from API, 0-indexed for list)
    page_num = citations[0].get("page", 1) if citations else 1
    page_idx = max(0, page_num - 1)
    page_idx = min(page_idx, len(pdf_pages) - 1)

    annotated = draw_citation_boxes(pdf_pages[page_idx], field_name, citations)
    ax.imshow(annotated)
    ax.set_title(f"{field_name}\n= {str(extracted_val)[:40]}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.suptitle("Extract V2: Bounding Box Citations", fontsize=14, y=1.02)
plt.show()

---
## Section 5: Confidence Scores

Enable `confidence_scores` to get a per-field confidence value (0.0 to 1.0) indicating how certain the model is about each extraction.

In [ ]:
# Extract the receipt with confidence scores enabled
confidence_job = client.extract.create(
    document_input_value=file_ids["receipt"],
    configuration={
        "data_schema": receipt_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
        "confidence_scores": True,
    },
)

confidence_job = wait_for_completion(client, confidence_job.id)

# Get expanded metadata
confidence_detail = client.extract.get(
    confidence_job.id,
    expand=["extract_metadata"],
)

conf_metadata = confidence_detail.extract_metadata.field_metadata.document_metadata
print("--- Per-Field Confidence Scores ---\n")
print("Each field has: confidence, parsing_confidence, extraction_confidence\n")
for field_name, field_meta in conf_metadata.items():
    value = confidence_detail.extract_result.get(field_name)
    if isinstance(field_meta, dict) and "confidence" in field_meta:
        score = field_meta["confidence"]
        print(f"  {field_name}: {score:.4f}  (value: {str(value)[:50]})")
    elif isinstance(field_meta, list):
        # Array fields (e.g. items) have per-element, per-subfield scores
        print(f"  {field_name}: [array with {len(field_meta)} element(s)]")
        for i, elem in enumerate(field_meta[:2]):
            if isinstance(elem, dict):
                for subfield, sub_meta in elem.items():
                    if isinstance(sub_meta, dict) and "confidence" in sub_meta:
                        print(f"    [{i}].{subfield}: {sub_meta['confidence']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Build data for the chart (top-level scalar fields only)
field_names = []
scores = []
for field_name, field_meta in conf_metadata.items():
    if isinstance(field_meta, dict) and "confidence" in field_meta:
        field_names.append(field_name)
        scores.append(float(field_meta["confidence"]))

# Bar chart
colors = ["red" if s < 0.8 else "steelblue" for s in scores]

fig, ax = plt.subplots(figsize=(10, max(4, len(field_names) * 0.35)))
bars = ax.barh(field_names, scores, color=colors)
ax.axvline(x=0.8, color="orange", linestyle="--", label="Threshold (0.8)")
ax.set_xlabel("Confidence Score")
ax.set_title("Per-Field Confidence Scores (red = below 0.8)")
ax.set_xlim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Filter fields below threshold
threshold = 0.8
low_confidence = [
    (name, score) for name, score in zip(field_names, scores) if score < threshold
]

if low_confidence:
    print(f"Fields below {threshold} confidence:")
    for name, score in low_confidence:
        print(f"  {name}: {score:.2f}")
else:
    print(f"All fields are at or above {threshold} confidence.")

---
## Section 6: Agentic vs Cost Effective

Compare the two extraction tiers on a SaaS pitch deck slide. The `agentic` tier uses more credits but may produce better results on complex or ambiguous documents.

In [ ]:
# Load the SaaS slide schema
with open("test-files/slide/schema.json") as f:
    slide_schema = json.load(f)

print(f"Slide schema fields: {list(slide_schema['properties'].keys())}")

In [ ]:
# Display the SaaS slide so we can compare extraction results against the source
from pdf2image import convert_from_path
import matplotlib.pyplot as plt

slide_pages = convert_from_path("test-files/slide/saas_slide.pdf", dpi=150)
fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(slide_pages[0])
ax.set_title("Source Document: SaaS Pitch Deck Slide")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Run cost_effective extraction
t0 = time.time()
ce_job = client.extract.create(
    document_input_value=file_ids["slide"],
    configuration={
        "data_schema": slide_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
    },
)
ce_job = wait_for_completion(client, ce_job.id)
ce_time = time.time() - t0
print(f"Cost effective: {ce_time:.1f}s")

# Run agentic extraction
t0 = time.time()
ag_job = client.extract.create(
    document_input_value=file_ids["slide"],
    configuration={
        "data_schema": slide_schema,
        "extraction_target": "per_doc",
        "tier": "agentic",
    },
)
ag_job = wait_for_completion(client, ag_job.id)
ag_time = time.time() - t0
print(f"Agentic: {ag_time:.1f}s")

In [ ]:
# Side-by-side comparison
def flatten_result(result, prefix=""):
    """Flatten nested dicts into dot-notation keys."""
    items = {}
    if isinstance(result, dict):
        for k, v in result.items():
            key = f"{prefix}{k}" if not prefix else f"{prefix}.{k}"
            if isinstance(v, dict):
                items.update(flatten_result(v, key))
            else:
                items[key] = v
    return items

ce_flat = flatten_result(ce_job.extract_result)
ag_flat = flatten_result(ag_job.extract_result)

all_keys = sorted(set(list(ce_flat.keys()) + list(ag_flat.keys())))

print(f"{'Field':<40} {'Cost Effective':<30} {'Agentic':<30}")
print("-" * 100)
for key in all_keys:
    ce_val = str(ce_flat.get(key, "--"))[:28]
    ag_val = str(ag_flat.get(key, "--"))[:28]
    match = "" if ce_val == ag_val else " *"
    print(f"{key:<40} {ce_val:<30} {ag_val:<30}{match}")

print(f"\nTiming: cost_effective={ce_time:.1f}s, agentic={ag_time:.1f}s")
print("(* = values differ between tiers)")

---
## Section 7: Per-Page Extraction

Extract one result object per page instead of one per document. Useful for multi-page documents where each page has its own data.

In [ ]:
# Extract patent with per_page target (first 3 pages only to keep it fast)
perpage_job = client.extract.create(
    document_input_value=file_ids["patent"],
    configuration={
        "data_schema": patent_schema,
        "extraction_target": "per_page",
        "tier": "cost_effective",
        "target_pages": "1-3",
    },
)

perpage_job = wait_for_completion(client, perpage_job.id)

# extract_result is a list when using per_page
results = perpage_job.extract_result
print(f"Got {len(results)} page results (type: {type(results).__name__})\n")

for i, page_result in enumerate(results):
    print(f"=== Page {i + 1} ===")
    for field, value in page_result.items():
        if value is not None:
            print(f"  {field}: {str(value)[:60]}")
    print()

---
## Section 8: Advanced Options

Demonstrate `target_pages`, `system_prompt`, and `parse_tier` configuration options.

In [ ]:
# target_pages: only extract from the cover page
targeted_job = client.extract.create(
    document_input_value=file_ids["patent"],
    configuration={
        "data_schema": patent_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
        "target_pages": "1",  # Only extract from page 1
    },
)

targeted_job = wait_for_completion(client, targeted_job.id)
print("\n--- Page 1 only ---")
for field, value in targeted_job.extract_result.items():
    if value is not None:
        print(f"  {field}: {str(value)[:60]}")

In [ ]:
# system_prompt: guide extraction behavior with custom instructions
prompted_job = client.extract.create(
    document_input_value=file_ids["receipt"],
    configuration={
        "data_schema": receipt_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
        "system_prompt": (
            "All dollar amounts should be formatted as plain numbers without "
            "currency symbols. Dates should be in ISO 8601 format (YYYY-MM-DD)."
        ),
    },
)

prompted_job = wait_for_completion(client, prompted_job.id)
print("\n--- With system_prompt ---")
print(json.dumps(prompted_job.extract_result, indent=2))

In [ ]:
# parse_tier: control the parsing step before extraction
parsed_job = client.extract.create(
    document_input_value=file_ids["slide"],
    configuration={
        "data_schema": slide_schema,
        "extraction_target": "per_doc",
        "tier": "cost_effective",
        "parse_tier": "agentic",  # Use agentic parsing even with cost_effective extraction
    },
)

parsed_job = wait_for_completion(client, parsed_job.id)
print("\n--- With agentic parse_tier ---")
print(json.dumps(parsed_job.extract_result, indent=2))

---
## Section 9: Job Management

List, inspect, and delete extraction jobs.

In [ ]:
# List recent jobs (paginated)
jobs_page = client.extract.list(page_size=5)

print(f"{'Job ID':<30} {'Status':<12} {'Created At'}")
print("-" * 70)
for i, job in enumerate(jobs_page):
    if i >= 5:
        break
    print(f"{job.id:<30} {job.status:<12} {job.created_at}")

In [ ]:
# Get a job with expanded configuration
detailed_job = client.extract.get(
    receipt_job.id,
    expand=["configuration"],
)

print(f"Job ID: {detailed_job.id}")
print(f"Status: {detailed_job.status}")
print(f"Configuration:")
config = detailed_job.configuration
print(f"  tier: {config.tier}")
print(f"  extraction_target: {config.extraction_target}")
print(f"  cite_sources: {config.cite_sources}")
print(f"  confidence_scores: {config.confidence_scores}")
print(f"  schema fields: {list(config.data_schema.get('properties', {}).keys())[:5]}...")

In [ ]:
# Delete a job (we'll delete the targeted pages job from Section 8)
job_to_delete = targeted_job.id
print(f"Deleting job: {job_to_delete}")

client.extract.delete(job_to_delete)
print("Deleted successfully.")

---

## Summary

This walkthrough covered:

1. **Setup**: SDK installation, client initialization, file uploads
2. **Quick Start**: Simple receipt extraction with JSON Schema
3. **Complex Documents**: 15-field patent extraction with arrays and nested types
4. **Citations & Bounding Boxes**: Per-field source citations with visual overlay on PDF pages
5. **Confidence Scores**: Per-field confidence values with threshold filtering
6. **Tier Comparison**: Cost effective vs agentic extraction side-by-side
7. **Per-Page Extraction**: One result per page for multi-page documents
8. **Advanced Options**: target_pages, system_prompt, parse_tier
9. **Job Management**: List, inspect, and delete jobs

For more information, visit the [LlamaCloud documentation](https://docs.cloud.llamaindex.ai).